# Motor Overheating Prediction — v3 (Sophisticated Pipeline)

**Objective:** Detect fault labels (0/1) for 6 servo motors using temperature / position / voltage at 10 Hz.

**Metric:** F1-score (class 1) averaged across 6 motors.

---

### Key innovations vs v2
| # | Innovation |
|---|------------|
| 1 | Minimal preprocessing — physical clip + ffill only, no centering |
| 2 | Trust-filtering of additional_data (void mislabelled sequences) |
| 3 | Rich per-motor feature engineering (CUSUM, matched filter, slope asym, …) |
| 4 | Multi-scale matched filter (shape_match / shape_scale) |
| 5 | Cross-motor features (voltage deviation from peers, sum-of-others velocity) |
| 6 | Per-motor calibrated fault injection pool |
| 7 | RF / HGB model family (no XGBoost) with class-weight fade |
| 8 | Rate-matched prediction instead of threshold sweeping |
| 9 | Viterbi episode decoding for motors 2 and 6 |
|10 | Protected rates for hard motors (3, 5, 6) |

### Notebook sections
| Section | Content |
|---------|--------|
| § 1 | Environment verification |
| § 2 | Imports |
| § 3 | Helper functions (preprocessing, trust-filter) |
| § 4 | Feature engineering (shape filter + build_features_for_df) |
| § 5 | Injection pool functions |
| § 6 | Model config, rate functions, Viterbi, sample weights |
| § 7a | Load base training + test data (raw, minimal preprocessing) |
| § 7b | Load + trust-filter additional data |
| § 7c | Build features on combined training data |
| § 7d | Build features on test data |
| § 7e | Build injection pool (raw injection → then build features) |
| § 8 | train_motor() function definition |
| § 9–14 | Motor 1 through 6 (one cell each) |
| § 15 | Summary (model, rate, val F1 per motor) |
| § 16 | Submission |

## § 1 · Environment verification

Checks the Python kernel and that all required data paths exist before proceeding.

In [1]:
import sys, os

print(f'Python : {sys.version}')
print(f'Kernel : {sys.executable}')
print()

NOTEBOOK_ROOT   = os.getcwd()
TRAIN_PATH      = os.path.join(NOTEBOOK_ROOT, 'data/training_data') + os.sep
TEST_PATH       = os.path.join(NOTEBOOK_ROOT, 'data/testing_data')  + os.sep
SAMPLE_SUB_PATH = os.path.join(NOTEBOOK_ROOT, 'sample_submission.csv')
SUBMISSION_DIR  = os.path.join(NOTEBOOK_ROOT, 'submissions')
os.makedirs(SUBMISSION_DIR, exist_ok=True)
ADDITIONAL_BASE = os.path.join(NOTEBOOK_ROOT, 'data/additional_data')
UTILITY_PATH    = os.path.join(NOTEBOOK_ROOT, 'kaggle_data_challenge')

checks = {
    'utility.py'                  : os.path.join(UTILITY_PATH, 'utility.py'),
    'training_data/'              : TRAIN_PATH,
    'testing_data/'               : TEST_PATH,
    'sample_submission.csv'       : SAMPLE_SUB_PATH,
    'additional_data/ (opcional)' : ADDITIONAL_BASE,
}
all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    mark   = 'OK   ' if exists else 'FALTA'
    print(f'  {mark}  {label:38s}  {path}')
    if not exists and 'opcional' not in label:
        all_ok = False

if os.path.exists(ADDITIONAL_BASE):
    grupos = [d for d in os.listdir(ADDITIONAL_BASE)
              if os.path.isdir(os.path.join(ADDITIONAL_BASE, d))
              and not d.startswith('_tmp')]
    print(f'\n  Grupos en additional_data/ ({len(grupos)} encontrados):')
    for g in grupos:
        gp      = os.path.join(ADDITIONAL_BASE, g)
        subdirs = [d for d in os.listdir(gp) if os.path.isdir(os.path.join(gp, d))]
        has_xl  = os.path.exists(os.path.join(gp, 'Test conditions.xlsx'))
        has_cp  = os.path.exists(os.path.join(gp, 'Test conditions copy.xlsx'))
        xl_mark = 'xlsx OK' if has_xl else ('xlsx (copia)' if has_cp else 'SIN xlsx')
        print(f'    {g}  —  {len(subdirs)} secuencias  —  {xl_mark}')

print()
print('Entorno OK' if all_ok else 'ATENCION: faltan rutas criticas')

Python : 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
Kernel : c:\Users\oscar\Desktop\TDS INDUSTRY\.venv\Scripts\python.exe

  OK     utility.py                              c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\utility.py
  OK     training_data/                          c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\kaggle_data_challenge\training_data\
  OK     testing_data/                           c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\kaggle_data_challenge\testing_data\
  OK     sample_submission.csv                   c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\kaggle_data_challenge\sample_submission.csv
  OK     additional_data/ (opcional)             c:\Users\oscar\Desktop\TDS INDUSTRY\additional_data

  Grupos en additional_data/ (3 encontrados):
    additional_data_20240524_group_6  —  14 secuencias  —  xlsx OK
    additional_training_data_group_1  —  13 secuencias  —  xlsx OK
 

## § 2 · Imports

All imports for the pipeline, including RandomForestClassifier, HistGradientBoostingClassifier,
GroupKFold and numpy stride tricks for the matched filter.

In [2]:
import warnings, shutil
import numpy  as np
import pandas as pd

from numpy.lib.stride_tricks import sliding_window_view

from sklearn.ensemble        import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics         import f1_score

warnings.filterwarnings('ignore')

# Utility from competition
sys.path.insert(1, UTILITY_PATH)
from utility import read_all_test_data_from_path

print('All imports OK')

All imports OK


## § 3 · Helper functions

### `pre_processing_minimal`
Only physical clip + forward-fill. NO centering, NO derived features.
Robust baselines handle centering better.

### `trust_filter_additional`
For each (test_condition, motor) block that is labelled as faulty:
- Compute baseline = min temperature during normal periods in that sequence.
- If max fault temperature does NOT exceed baseline + margin → void the labels (set to 0).
- Returns (filtered_df, audit_df) showing kept / voided counts per block.

In [3]:
# ─────────────────────────────────────────────────────────────────
#  Minimal preprocessing callback (used by read_all_test_data_from_path)
# ─────────────────────────────────────────────────────────────────
def pre_processing_minimal(df: pd.DataFrame):
    """Physical clip + ffill only. No centering, no feature derivation."""
    if len(df) == 0:
        return
    df['temperature'] = (
        df['temperature']
        .where(df['temperature'].between(0, 100), np.nan)
        .ffill()
    )
    df['voltage'] = (
        df['voltage']
        .where(df['voltage'].between(6000, 9000), np.nan)
        .ffill()
    )
    df['position'] = (
        df['position']
        .where(df['position'].between(0, 1000), np.nan)
        .ffill()
    )


# ─────────────────────────────────────────────────────────────────
#  Trust-filter for additional_data
# ─────────────────────────────────────────────────────────────────
def trust_filter_additional(df: pd.DataFrame, temp_rise_margin: float = 1.0):
    """
    For each (test_condition, motor) block labelled as faulty:
      - baseline = min temperature during *normal* (label=0) periods in the same sequence.
      - If max fault temperature <= baseline + temp_rise_margin  →  void labels (set 0).

    Returns
    -------
    filtered_df : DataFrame with voided labels set to 0
    audit_df    : DataFrame with columns [test_condition, motor_id, kept, voided]
    """
    df = df.copy()
    audit_rows = []

    for seq in df['test_condition'].unique():
        seq_mask = df['test_condition'] == seq
        seq_df   = df[seq_mask]

        for mid in range(1, 7):
            label_col = f'data_motor_{mid}_label'
            temp_col  = f'data_motor_{mid}_temperature'

            if label_col not in seq_df.columns or temp_col not in seq_df.columns:
                continue

            labels = seq_df[label_col]
            temps  = seq_df[temp_col]

            # Only process if there are labelled faults
            if (labels == 1).sum() == 0:
                continue

            normal_temps = temps[labels == 0]
            if len(normal_temps) == 0:
                continue
            baseline     = float(normal_temps.min())
            fault_temps  = temps[labels == 1]
            max_fault_t  = float(fault_temps.max())

            if max_fault_t <= baseline + temp_rise_margin:
                # Void the labels — mislabelled sequence
                df.loc[seq_mask & (df[label_col] == 1), label_col] = 0
                audit_rows.append({
                    'test_condition': seq, 'motor_id': mid,
                    'kept': 0, 'voided': int((labels == 1).sum()),
                    'baseline': round(baseline, 3),
                    'max_fault_temp': round(max_fault_t, 3),
                })
            else:
                audit_rows.append({
                    'test_condition': seq, 'motor_id': mid,
                    'kept': int((labels == 1).sum()), 'voided': 0,
                    'baseline': round(baseline, 3),
                    'max_fault_temp': round(max_fault_t, 3),
                })

    audit_df = pd.DataFrame(audit_rows)
    return df, audit_df


# ─────────────────────────────────────────────────────────────────
#  Group loader (same logic as v2 but uses pre_processing_minimal)
# ─────────────────────────────────────────────────────────────────
def _validate_sequence(seq_path):
    csvs = [f for f in os.listdir(seq_path) if f.endswith('.csv')]
    if not csvs:
        return False
    for csv_file in csvs:
        try:
            lines = [l for l in open(os.path.join(seq_path, csv_file)).readlines() if l.strip()]
            if len(lines) <= 1:
                return False
        except Exception:
            return False
    return True


def _load_group_raw(group_path, group_name):
    """Load one additional_data group with minimal preprocessing, return raw DataFrame or None."""
    xlsx      = os.path.join(group_path, 'Test conditions.xlsx')
    xlsx_copy = os.path.join(group_path, 'Test conditions copy.xlsx')
    if not os.path.exists(xlsx) and os.path.exists(xlsx_copy):
        try:
            shutil.copy(xlsx_copy, xlsx)
        except Exception:
            pass
    if not os.path.exists(xlsx):
        return None

    tmp = os.path.join(ADDITIONAL_BASE, f'_tmp_{group_name}')
    try:
        if os.path.exists(tmp):
            shutil.rmtree(tmp)
    except Exception:
        pass
    os.makedirs(tmp, exist_ok=True)
    shutil.copy(xlsx, os.path.join(tmp, 'Test conditions.xlsx'))

    subdirs = [d for d in os.listdir(group_path) if os.path.isdir(os.path.join(group_path, d))]
    n_ok = 0
    for sd in subdirs:
        sp  = os.path.join(group_path, sd)
        dst = os.path.join(tmp, sd)
        if _validate_sequence(sp):
            shutil.copytree(sp, dst, dirs_exist_ok=True)
            n_ok += 1

    result = None
    if n_ok > 0:
        try:
            result = read_all_test_data_from_path(tmp + os.sep, pre_processing_minimal, is_plot=False)
        except Exception as e:
            print(f'  [error loading {group_name}]: {e}')

    try:
        shutil.rmtree(tmp)
    except Exception:
        pass
    return result


print('Helper functions defined')

Helper functions defined


## § 4 · Feature engineering

### `_matched_filter(elev_arr, scales)`
Multi-scale matched filter using `numpy.lib.stride_tricks.sliding_window_view`.
For each scale L: builds a rise-decay template (rise L//3, fall 2L//3), zero-mean unit-norm.
Returns (best_corr, best_scale) per timestep.

### `_motor_features(temp, pos, volt, include_shape)`
All per-motor time-series features: deltas, rates, rolling stats, drift, CUSUM, run-length,
slope asymmetry, and (optionally) matched-filter outputs.

### `build_features_for_df(df)`
Operates on the wide DataFrame (all motors).
Groups by test_condition, calls `_motor_features` per sequence per motor,
then adds cross-motor features (voltage deviation from peers, sum-of-others velocity, etc.).

In [4]:
# ─────────────────────────────────────────────────────────────────
#  Multi-scale matched filter
# ─────────────────────────────────────────────────────────────────
def _matched_filter(elev_arr: np.ndarray, scales=(15, 30, 60, 120)):
    """
    Multi-scale matched filter on elevation signal.
    For each scale L:
      - Build rise-decay template: rise over L//3, fall over 2L//3
      - Zero-mean, unit-norm normalise template
      - Slide window of size L over elev_arr (padded with L-1 zeros)
      - Normalise each window (mean/std), compute dot product → clip at 0
    Returns (best_corr, best_scale) arrays — max across scales per timestep.
    """
    n = len(elev_arr)
    best_corr  = np.zeros(n, dtype=np.float32)
    best_scale = np.zeros(n, dtype=np.float32)

    for L in scales:
        n_rise = L // 3
        n_fall = L - n_rise
        tmpl   = np.concatenate([np.linspace(0, 1, n_rise), np.linspace(1, 0, n_fall)])
        tmpl   = tmpl - tmpl.mean()
        norm_t = np.linalg.norm(tmpl)
        if norm_t < 1e-8:
            continue
        tmpl = tmpl / norm_t

        # Pad signal so output length == n
        padded = np.concatenate([np.zeros(L - 1, dtype=np.float32), elev_arr.astype(np.float32)])
        # shape: (n, L)
        windows = sliding_window_view(padded, window_shape=L)

        w_mean = windows.mean(axis=1, keepdims=True)
        w_std  = windows.std(axis=1, keepdims=True) + 1e-8
        w_norm = (windows - w_mean) / w_std

        corr = np.clip(w_norm @ tmpl, 0, None)  # shape (n,)

        mask = corr > best_corr
        best_corr[mask]  = corr[mask]
        best_scale[mask] = float(L)

    return best_corr, best_scale


# ─────────────────────────────────────────────────────────────────
#  Per-motor feature dict
# ─────────────────────────────────────────────────────────────────
def _motor_features(temp: pd.Series, pos: pd.Series, volt: pd.Series,
                    include_shape: bool = True) -> dict:
    """
    Compute per-motor features.  All outputs are numpy arrays of len(temp).

    Parameters
    ----------
    temp, pos, volt   : per-motor Series (already clipped / ffilled)
    include_shape     : whether to run the matched filter

    Returns
    -------
    dict of {feature_name: np.ndarray}
    """
    n = len(temp)
    out = {}

    # Basic deltas / rates
    out['temp_delta']  = (temp - temp.iloc[0]).values.astype(np.float32)
    velocity           = pos.diff(1).fillna(0)
    acceleration       = velocity.diff(1).fillna(0)
    temp_rate          = temp.diff(1).fillna(0)
    temp_accel         = temp_rate.diff(1).fillna(0)
    volt_rate          = volt.diff(1).fillna(0)

    out['velocity']    = velocity.values.astype(np.float32)
    out['acceleration']= acceleration.values.astype(np.float32)
    out['temp_rate']   = temp_rate.values.astype(np.float32)
    out['temp_accel']  = temp_accel.values.astype(np.float32)
    out['volt_rate']   = volt_rate.values.astype(np.float32)

    # Rolling std features
    out['roll_std_pos_50']  = pos.rolling(50,  min_periods=1).std().fillna(0).values.astype(np.float32)
    out['roll_std_temp_50'] = temp.rolling(50, min_periods=1).std().fillna(0).values.astype(np.float32)
    out['roll_std_volt_50'] = volt.rolling(50, min_periods=1).std().fillna(0).values.astype(np.float32)

    # Temperature drift vs rolling mean
    out['temp_drift200'] = (
        temp - temp.rolling(200, min_periods=1).mean()
    ).values.astype(np.float32)

    # Is-moving (smoothed)
    out['is_moving'] = (
        (velocity.abs() > 0.5).astype(float)
        .rolling(20, min_periods=1).mean()
    ).values.astype(np.float32)

    # Temperature above rolling min-200
    out['temp_above_min200'] = (
        temp - temp.rolling(200, min_periods=1).min()
    ).values.astype(np.float32)

    # Temperature range over rolling 200
    roll200_max = temp.rolling(200, min_periods=1).max()
    roll200_min = temp.rolling(200, min_periods=1).min()
    out['temp_range200'] = (roll200_max - roll200_min).values.astype(np.float32)

    # Temperature above rolling 10th-percentile over 600 samples
    roll600_q10 = temp.rolling(600, min_periods=10).quantile(0.10)
    # fill early NaNs with expanding quantile
    exp_q10     = temp.expanding(min_periods=5).quantile(0.10)
    qbase       = roll600_q10.fillna(exp_q10)
    out['temp_above_qbase'] = (temp - qbase).fillna(0).values.astype(np.float32)

    # Temperature above expanding 10th-percentile
    out['temp_above_expq'] = (
        temp - temp.expanding(min_periods=5).quantile(0.10)
    ).fillna(0).values.astype(np.float32)

    # Temperature above cumulative minimum  (elevation)
    elev_cummin = (temp - temp.expanding().min()).fillna(0)
    out['temp_above_cummin'] = elev_cummin.values.astype(np.float32)

    # Normalised elevation
    first50_std = float(temp.iloc[:50].std()) if n >= 50 else float(temp.std())
    if np.isnan(first50_std) or first50_std < 1e-6:
        first50_std = 1e-6
    elev_norm = elev_cummin / (first50_std + 1e-6)
    out['temp_elev_norm'] = elev_norm.values.astype(np.float32)

    # One-sided CUSUM on elev_cummin
    slack = 0.5
    elev_arr = elev_cummin.values.astype(np.float64)
    cusum = np.zeros(n, dtype=np.float32)
    s = 0.0
    for i in range(n):
        if elev_arr[i] < slack:
            s = 0.0
        else:
            s = s + elev_arr[i] - slack
        cusum[i] = s
    out['cusum_up'] = cusum

    # Consecutive run-length where elev_cummin >= 1.0
    elev_run = np.zeros(n, dtype=np.float32)
    run = 0
    for i in range(n):
        if elev_arr[i] >= 1.0:
            run += 1
        else:
            run = 0
        elev_run[i] = run
    out['elev_runlen'] = elev_run

    # Slope asymmetry (lag=15)
    lag = 15
    elev_s = elev_cummin.values.astype(np.float32)
    slope_asym = np.zeros(n, dtype=np.float32)
    for i in range(2 * lag, n):
        left  = max(0.0, elev_s[i - lag]   - elev_s[i - 2 * lag])
        right = max(0.0, elev_s[i - lag]   - elev_s[i])
        slope_asym[i] = left * right
    out['slope_asym'] = slope_asym

    # Multi-scale matched filter (optional)
    if include_shape:
        shape_match, shape_scale = _matched_filter(elev_arr, scales=(15, 30, 60, 120))
        out['shape_match'] = shape_match.astype(np.float32)
        out['shape_scale'] = shape_scale.astype(np.float32)

    return out


# ─────────────────────────────────────────────────────────────────
#  Main feature-build function
# ─────────────────────────────────────────────────────────────────
SHAPE_MOTORS = {1, 2, 3, 5}   # include_shape=True for these motors

def build_features_for_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build rich per-motor + cross-motor features on a WIDE DataFrame.
    Groups by test_condition, processes each sequence independently.
    Writes new columns in-place and returns the enriched DataFrame.
    """
    df = df.copy()
    sequences = df['test_condition'].unique()
    n_seq = len(sequences)
    print(f'  Building features for {n_seq} sequences ...')

    for seq_idx, seq in enumerate(sequences):
        if (seq_idx + 1) % 10 == 0 or (seq_idx + 1) == n_seq:
            print(f'  ... {seq_idx + 1}/{n_seq}')
        mask = df['test_condition'] == seq
        idx  = df.index[mask]

        per_motor_feats = {}  # mid -> {feat_name -> arr}

        # ── Per-motor pass ───────────────────────────────────────
        for mid in range(1, 7):
            tc  = f'data_motor_{mid}_temperature'
            pc  = f'data_motor_{mid}_position'
            vc  = f'data_motor_{mid}_voltage'

            if tc not in df.columns:
                continue

            temp = df.loc[mask, tc]
            pos  = df.loc[mask, pc] if pc in df.columns else pd.Series(np.zeros(mask.sum()), index=idx)
            volt = df.loc[mask, vc] if vc in df.columns else pd.Series(np.zeros(mask.sum()), index=idx)

            include_shape = (mid in SHAPE_MOTORS)
            feats = _motor_features(temp, pos, volt, include_shape=include_shape)
            per_motor_feats[mid] = feats

            for feat_name, arr in feats.items():
                col = f'data_motor_{mid}_{feat_name}'
                df.loc[mask, col] = arr

        # ── Cross-motor features ─────────────────────────────────
        # mean voltage across all 6 motors
        volt_cols = [f'data_motor_{mid}_voltage' for mid in range(1, 7)
                     if f'data_motor_{mid}_voltage' in df.columns]
        if volt_cols:
            mean_voltage = df.loc[mask, volt_cols].mean(axis=1)
            df.loc[mask, 'mean_voltage'] = mean_voltage.values

            for mid in range(1, 7):
                vc = f'data_motor_{mid}_voltage'
                if vc in df.columns:
                    df.loc[mask, f'data_motor_{mid}_volt_dev_peers'] = (
                        df.loc[mask, vc].values - mean_voltage.values
                    )

        # sum of other 5 motors' velocity
        for mid in range(1, 7):
            other_vel_cols = [
                f'data_motor_{m}_velocity' for m in range(1, 7)
                if m != mid and f'data_motor_{m}_velocity' in df.columns
            ]
            if other_vel_cols:
                df.loc[mask, f'data_motor_{mid}_other_vel_sum'] = (
                    df.loc[mask, other_vel_cols].fillna(0).sum(axis=1).values
                )

        # count of other motors that are 'moving'
        for mid in range(1, 7):
            other_mv_cols = [
                f'data_motor_{m}_is_moving' for m in range(1, 7)
                if m != mid and f'data_motor_{m}_is_moving' in df.columns
            ]
            if other_mv_cols:
                df.loc[mask, f'data_motor_{mid}_other_moving_count'] = (
                    df.loc[mask, other_mv_cols].fillna(0).sum(axis=1).values
                )

    df.fillna(0, inplace=True)
    return df


print('Feature engineering functions defined')

Feature engineering functions defined


## § 5 · Injection pool functions

### `make_injection_pool(train_df_raw, rng)`
For each motor:
- Find sequences that are entirely normal for that motor.
- Create `n` synthetic sequences by picking a random normal sequence,
  then injecting a rise-decay temperature pattern at a random location.
- Other motors' labels are set to 0 (synthetic fault is motor-specific).
- Sequences are given unique IDs like `syn_{motor_id}_{i}`.

Returns `dict {motor_id: raw_DataFrame}`.  Features are built *after* injection.

In [5]:
# Per-motor injection parameters
PER_MOTOR_INJECT = {
    1: dict(n=4,  peak_low=2, peak_high=10, dur_low=120, dur_high=400),
    2: dict(n=4,  peak_low=2, peak_high=10, dur_low=120, dur_high=400),
    3: dict(n=16, peak_low=1, peak_high=4,  dur_low=40,  dur_high=500),  # subtle
    4: dict(n=4,  peak_low=2, peak_high=10, dur_low=120, dur_high=400),
    5: dict(n=16, peak_low=1, peak_high=4,  dur_low=40,  dur_high=500),  # subtle
    6: dict(n=4,  peak_low=2, peak_high=10, dur_low=120, dur_high=400),
}


def _inject_rise_decay(temp_series: pd.Series, label_series: pd.Series,
                        start: int, duration: int, peak: float) -> pd.Series:
    """
    Inject a rise-decay temperature episode at position [start, start+duration).
    Returns modified temperature Series.
    """
    temp = temp_series.copy()
    n    = len(temp)
    end  = min(start + duration, n)
    dur  = end - start
    if dur <= 0:
        return temp

    n_rise = dur // 3
    n_fall = dur - n_rise
    t0     = float(temp.iloc[start])
    peak_t = t0 + peak

    rise = np.linspace(t0, peak_t, n_rise + 1)[:-1]
    fall = np.linspace(peak_t, t0,  n_fall)
    bump = np.concatenate([rise, fall])

    iloc_positions = list(range(start, end))
    temp.iloc[iloc_positions] = bump
    return temp


def make_injection_pool(train_df_raw: pd.DataFrame,
                        rng: np.random.Generator = None) -> dict:
    """
    Build per-motor injection pools from raw (no feature) training data.

    Returns
    -------
    dict {motor_id: DataFrame}  — each DataFrame is a stack of synthetic
    sequences, RAW (features not yet built).
    """
    if rng is None:
        rng = np.random.default_rng(42)

    pool = {}
    all_sequences = train_df_raw['test_condition'].unique()

    for mid in range(1, 7):
        cfg       = PER_MOTOR_INJECT[mid]
        n_syn     = cfg['n']
        label_col = f'data_motor_{mid}_label'
        temp_col  = f'data_motor_{mid}_temperature'

        if label_col not in train_df_raw.columns:
            continue

        # Find sequences that are entirely normal for this motor
        normal_seqs = []
        for seq in all_sequences:
            seq_df = train_df_raw[train_df_raw['test_condition'] == seq]
            if (seq_df[label_col] == 1).sum() == 0:
                normal_seqs.append(seq)

        if len(normal_seqs) == 0:
            print(f'  Motor {mid}: no normal sequences found, skipping injection.')
            continue

        syn_dfs = []
        for i in range(n_syn):
            # Pick a random normal sequence
            chosen_seq  = normal_seqs[int(rng.integers(0, len(normal_seqs)))]
            base_df     = train_df_raw[train_df_raw['test_condition'] == chosen_seq].copy()
            base_df     = base_df.reset_index(drop=True)
            seq_len     = len(base_df)

            # Random duration and start
            duration = int(rng.integers(cfg['dur_low'], cfg['dur_high'] + 1))
            peak     = float(rng.uniform(cfg['peak_low'], cfg['peak_high']))
            max_start = max(1, seq_len - duration)
            start    = int(rng.integers(0, max_start))

            # Inject temperature rise-decay
            if temp_col in base_df.columns:
                base_df[temp_col] = _inject_rise_decay(
                    base_df[temp_col], base_df[label_col], start, duration, peak
                )

            # Set labels for this motor in the injected window
            base_df[label_col] = 0
            end = min(start + duration, seq_len)
            base_df.loc[start:end - 1, label_col] = 1

            # Void other motors' labels (synthetic fault only for target motor)
            for other_mid in range(1, 7):
                if other_mid == mid:
                    continue
                other_lc = f'data_motor_{other_mid}_label'
                if other_lc in base_df.columns:
                    base_df[other_lc] = 0

            # Unique test_condition ID
            base_df['test_condition'] = f'syn_{mid}_{i}'
            syn_dfs.append(base_df)

        if syn_dfs:
            pool[mid] = pd.concat(syn_dfs, ignore_index=True)
            n1_total  = (pool[mid][label_col] == 1).sum()
            print(f'  Motor {mid}: {n_syn} synthetic sequences, {n1_total} fault rows')

    return pool


print('Injection pool functions defined')

Injection pool functions defined


## § 6 · Model config, rate functions, Viterbi, sample weights

### Model family
- Motors 1–4 → RandomForest (RF)
- Motors 5–6 → HistGradientBoosting (HGB)

### Sample weights with prevalence fade
`compute_weights` fades the class imbalance weight down as prevalence rises — avoids over-weighting when data is balanced.

### Rate-matched prediction
Instead of sweeping probability thresholds, find the **rate** (fraction of positive predictions)
that maximises F1 on OOF, then apply the same rate to test predictions.

### Viterbi episode decoding (motors 2 and 6)
2-state HMM in log-space.  `decode_to_rate` bisects on bias to hit a target positive rate.

In [6]:
# ─────────────────────────────────────────────────────────────────
#  Model definitions
# ─────────────────────────────────────────────────────────────────
FINAL_MODELS = {1: 'RF', 2: 'RF', 3: 'RF', 4: 'RF', 5: 'HGB', 6: 'HGB'}

def _make_model(algo: str):
    if algo == 'RF':
        return RandomForestClassifier(
            n_estimators=150,
            min_samples_leaf=10,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=42,
        )
    elif algo == 'HGB':
        return HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            min_samples_leaf=100,
            l2_regularization=5.0,
            random_state=42,
        )
    else:
        raise ValueError(f'Unknown algo: {algo}')


# ─────────────────────────────────────────────────────────────────
#  Sample weights with prevalence fade
# ─────────────────────────────────────────────────────────────────
WEIGHT_FADE_PREVALENCE = 0.15

def compute_weights(y: np.ndarray) -> np.ndarray:
    """
    Compute per-sample weights with prevalence-based fade.
    As prevalence rises toward WEIGHT_FADE_PREVALENCE, the positive
    class weight fades toward 1.0 (no weighting needed).
    """
    n_neg = (y == 0).sum()
    n_pos = (y == 1).sum()
    if n_pos == 0 or n_neg == 0:
        return np.ones(len(y), dtype=np.float32)

    prevalence    = n_pos / len(y)
    raw_weight    = np.sqrt(n_neg / n_pos)
    fade          = min(1.0, prevalence / WEIGHT_FADE_PREVALENCE)
    eff_weight    = 1.0 + (raw_weight - 1.0) * (1.0 - fade)
    return np.where(y == 1, eff_weight, 1.0).astype(np.float32)


# ─────────────────────────────────────────────────────────────────
#  Rate-matched prediction
# ─────────────────────────────────────────────────────────────────
def rate_matched_threshold(probs: np.ndarray, target_rate: float) -> float:
    """
    Find the probability threshold that yields target_rate positive predictions.
    Sorts probs descending and takes the n_positive-th value as threshold.
    """
    sorted_probs = np.sort(probs)[::-1]
    n_positive   = max(1, int(target_rate * len(probs)))
    if n_positive < len(sorted_probs):
        return float(sorted_probs[n_positive])
    return 0.0


def compute_additional_prevalence(additional_df: pd.DataFrame, motor_id: int) -> float:
    """
    Compute fraction of label=1 rows for motor_id in the trust-filtered additional_data.
    """
    label_col = f'data_motor_{motor_id}_label'
    if label_col not in additional_df.columns or len(additional_df) == 0:
        return 0.01
    n1 = (additional_df[label_col] == 1).sum()
    return float(n1) / len(additional_df)


def find_oof_optimal_rate(oof_probs: np.ndarray, oof_labels: np.ndarray) -> float:
    """
    Sweep rates from 0.001 to 0.30 in 100 steps.
    Return the rate that maximises F1 on OOF predictions.
    """
    rates       = np.linspace(0.001, 0.30, 100)
    best_rate   = 0.05
    best_f1     = -1.0
    for rate in rates:
        thr  = rate_matched_threshold(oof_probs, rate)
        pred = (oof_probs >= thr).astype(int)
        f1   = f1_score(oof_labels, pred, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1  = f1
            best_rate = rate
    return best_rate


# ─────────────────────────────────────────────────────────────────
#  Viterbi episode decoding
# ─────────────────────────────────────────────────────────────────
EPISODE_MOTORS = {2, 6}

def mean_episode_length(labels: np.ndarray) -> float:
    """Mean length of contiguous runs of 1s.  Default 50 if no episodes."""
    if labels.sum() == 0:
        return 50.0
    lengths = []
    run = 0
    for v in labels:
        if v == 1:
            run += 1
        elif run > 0:
            lengths.append(run)
            run = 0
    if run > 0:
        lengths.append(run)
    return float(np.mean(lengths)) if lengths else 50.0


def viterbi_path(prob: np.ndarray, a11: float = 0.99,
                  a00: float = 0.999, bias: float = 0.0) -> np.ndarray:
    """
    2-state HMM Viterbi decoder in log-space.
    States: 0 = normal, 1 = fault.
    Transition matrix: [[a00, 1-a00], [1-a11, a11]]
    Emission: p_fault = clip(prob + bias, 1e-6, 1-1e-6)
    """
    n     = len(prob)
    p_f   = np.clip(prob + bias, 1e-6, 1 - 1e-6)
    p_n   = 1.0 - p_f

    log_a = np.log(np.array([[a00,     1 - a00],
                              [1 - a11, a11    ]]))

    # log-emission[t, state]
    log_emit = np.column_stack([np.log(p_n), np.log(p_f)])

    delta  = np.full((n, 2), -np.inf)
    psi    = np.zeros((n, 2), dtype=int)

    # Initialise: equal prior
    delta[0] = np.log(0.5) + log_emit[0]

    for t in range(1, n):
        for s in range(2):
            candidates = delta[t - 1] + log_a[:, s]
            psi[t, s]  = int(np.argmax(candidates))
            delta[t, s]= candidates[psi[t, s]] + log_emit[t, s]

    # Backtrack
    path = np.zeros(n, dtype=int)
    path[-1] = int(np.argmax(delta[-1]))
    for t in range(n - 2, -1, -1):
        path[t] = psi[t + 1, path[t + 1]]

    return path


def decode_all(probs_arr: np.ndarray, seq_ids_arr: np.ndarray,
               a11: float, a00: float, bias: float = 0.0) -> np.ndarray:
    """Apply viterbi_path per sequence, return full predictions array."""
    preds    = np.zeros(len(probs_arr), dtype=int)
    seq_ids  = np.unique(seq_ids_arr)
    for sid in seq_ids:
        mask = (seq_ids_arr == sid)
        preds[mask] = viterbi_path(probs_arr[mask], a11=a11, a00=a00, bias=bias)
    return preds


def decode_to_rate(probs_arr: np.ndarray, seq_ids_arr: np.ndarray,
                   target_rate: float, a11: float, a00: float) -> np.ndarray:
    """
    Bisect on bias (range -10 to 10) so that Viterbi decoded positive rate
    matches target_rate.  50 bisection iterations.
    """
    lo, hi = -10.0, 10.0
    for _ in range(50):
        mid_bias = 0.5 * (lo + hi)
        preds    = decode_all(probs_arr, seq_ids_arr, a11, a00, bias=mid_bias)
        rate     = preds.mean()
        if rate < target_rate:
            lo = mid_bias
        else:
            hi = mid_bias
    return decode_all(probs_arr, seq_ids_arr, a11, a00, bias=0.5 * (lo + hi))


# ─────────────────────────────────────────────────────────────────
#  Protected rates and test-rate cap
# ─────────────────────────────────────────────────────────────────
PROTECTED_RATES   = {3: 0.005, 5: 0.004, 6: 0.008}
TEST_RATE_CAP_MULT = 2.0

# Results store
models = {}

print('Model config, rate functions, and Viterbi decoder defined')

Model config, rate functions, and Viterbi decoder defined


## § 7a · Load base training + test data (raw, minimal preprocessing)

Uses `pre_processing_minimal` — physical clip + ffill only, no centering, no feature derivation.
Features will be built later in § 7c / § 7d.

In [7]:
print('=== Loading base training data (minimal preprocessing) ===')
train_raw = read_all_test_data_from_path(TRAIN_PATH, pre_processing_minimal, is_plot=False)
print(f'  Shape: {train_raw.shape}')
print(f'  Sequences: {train_raw["test_condition"].nunique()}')

print()
print('=== Loading test data (minimal preprocessing) ===')
test_raw = read_all_test_data_from_path(TEST_PATH, pre_processing_minimal, is_plot=False)
print(f'  Shape: {test_raw.shape}')
print(f'  Sequences: {test_raw["test_condition"].nunique()}')

print()
print('=== Label distribution (base training only) ===')
print(f'{"Motor":<8} {"Normal":>8} {"Fault":>8} {"% fault":>9}')
print('-' * 38)
for mid in range(1, 7):
    col = f'data_motor_{mid}_label'
    if col in train_raw.columns:
        c0  = (train_raw[col] == 0).sum()
        c1  = (train_raw[col] == 1).sum()
        pct = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
        flag = '  <- imbalanced' if pct < 1.0 else ''
        print(f'{mid:<8} {c0:>8} {c1:>8} {pct:>8.1f}%{flag}')

=== Loading base training data (minimal preprocessing) ===
  Shape: (39309, 26)
  Sequences: 23

=== Loading test data (minimal preprocessing) ===
  Shape: (14157, 26)
  Sequences: 8

=== Label distribution (base training only) ===
Motor      Normal    Fault   % fault
--------------------------------------
1           37960     1349      3.4%
2           32577     6732     17.1%
3           39182      127      0.3%  <- imbalanced
4           32570     6739     17.1%
5           39125      184      0.5%  <- imbalanced
6           37377     1932      4.9%


## § 7b · Load + trust-filter additional data

Scans `additional_data/` for all groups (skipping `_tmp_` dirs).
Each group is loaded with `pre_processing_minimal`, then passed through `trust_filter_additional`
which voids mislabelled fault sequences where the temperature signature is not distinguishable
from the normal baseline.

In [8]:
additional_dfs = []   # list of trust-filtered additional DataFrames
all_audit_rows = []   # audit rows from trust_filter_additional

if not os.path.exists(ADDITIONAL_BASE):
    print(f'additional_data/ not found at {ADDITIONAL_BASE} — skipping.')
else:
    grupos = sorted([
        d for d in os.listdir(ADDITIONAL_BASE)
        if os.path.isdir(os.path.join(ADDITIONAL_BASE, d))
        and not d.startswith('_tmp')
    ])
    print(f'Found {len(grupos)} groups in additional_data/')

    for grupo in grupos:
        gpath = os.path.join(ADDITIONAL_BASE, grupo)
        print(f'\n  [{grupo}]')
        raw_g = _load_group_raw(gpath, grupo)
        if raw_g is None:
            print('    -> skipped (no xlsx or no valid sequences)')
            continue
        print(f'    Loaded: {raw_g.shape[0]} rows, {raw_g["test_condition"].nunique()} sequences')

        filtered_g, audit_g = trust_filter_additional(raw_g, temp_rise_margin=1.0)
        if len(audit_g) > 0:
            voided_blocks = (audit_g['voided'] > 0).sum()
            kept_blocks   = (audit_g['kept'] > 0).sum()
            print(f'    Trust-filter: {kept_blocks} blocks kept, {voided_blocks} blocks voided')
        all_audit_rows.append(audit_g)
        additional_dfs.append(filtered_g)

    if additional_dfs:
        additional_df = pd.concat(additional_dfs, ignore_index=True)
        print(f'\n  Combined additional_data: {additional_df.shape[0]} rows, '
              f'{additional_df["test_condition"].nunique()} sequences')

        print()
        print('  Label distribution in trust-filtered additional_data:')
        print(f'  {"Motor":<8} {"Normal":>8} {"Fault":>8} {"% fault":>9}')
        print('  ' + '-' * 38)
        for mid in range(1, 7):
            col = f'data_motor_{mid}_label'
            if col in additional_df.columns:
                c0  = (additional_df[col] == 0).sum()
                c1  = (additional_df[col] == 1).sum()
                pct = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
                print(f'  {mid:<8} {c0:>8} {c1:>8} {pct:>8.2f}%')
    else:
        additional_df = pd.DataFrame()
        print('\nNo additional data loaded.')

audit_df = pd.concat(all_audit_rows, ignore_index=True) if all_audit_rows else pd.DataFrame()
print('\nDone.')

Found 3 groups in additional_data/

  [additional_data_20240524_group_6]
    Loaded: 26086 rows, 13 sequences
    Trust-filter: 8 blocks kept, 0 blocks voided

  [additional_training_data_group_1]
    Loaded: 27592 rows, 13 sequences
    Trust-filter: 4 blocks kept, 0 blocks voided

  [additional_training_data_group_7]
    Loaded: 6420 rows, 10 sequences
    Trust-filter: 10 blocks kept, 0 blocks voided

  Combined additional_data: 60098 rows, 36 sequences

  Label distribution in trust-filtered additional_data:
  Motor      Normal    Fault   % fault
  --------------------------------------
  1           57632     2466     4.10%
  2           58734     1364     2.27%
  3           59391      707     1.18%
  4           58163     1935     3.22%
  5           58660     1438     2.39%
  6           59073     1025     1.71%

Done.


## § 7c · Build features on combined training data

Concatenates base training data + trust-filtered additional data,
then runs `build_features_for_df` to add all engineered features.
The result is stored in `train_feat_df`.

In [9]:
print('Combining base training + additional data ...')
if len(additional_df) > 0:
    combined_raw = pd.concat([train_raw, additional_df], ignore_index=True)
else:
    combined_raw = train_raw.copy()

print(f'Combined raw shape: {combined_raw.shape}  '
      f'({combined_raw["test_condition"].nunique()} sequences)')
print()
print('Building features for training data ...')
train_feat_df = build_features_for_df(combined_raw)
print(f'Train feature DataFrame shape: {train_feat_df.shape}')
print('Done.')

Combining base training + additional data ...
Combined raw shape: (99407, 26)  (59 sequences)

Building features for training data ...
  Building features for 59 sequences ...
  ... 10/59
  ... 20/59
  ... 30/59
  ... 40/59
  ... 50/59
  ... 59/59
Train feature DataFrame shape: (99407, 173)
Done.


## § 7d · Build features on test data

Applies the same feature engineering pipeline to the test set.
Stored in `test_feat_df`.

In [10]:
print('Building features for test data ...')
test_feat_df = build_features_for_df(test_raw)
print(f'Test feature DataFrame shape: {test_feat_df.shape}')
print(f'Test sequences: {test_feat_df["test_condition"].nunique()}')
print('Done.')

Building features for test data ...
  Building features for 8 sequences ...
  ... 8/8
Test feature DataFrame shape: (14157, 173)
Test sequences: 8
Done.


## § 7e · Build injection pool

1. `make_injection_pool` creates synthetic fault sequences from the **raw** training data.
2. `build_features_for_df` is applied to each motor's injection pool to get `injection_pool_feat`.

The injection pool is built from raw data (before feature engineering) and features are computed
afterwards to maintain consistency with the training pipeline.

In [11]:
print('Building raw injection pool from training data ...')
rng_global  = np.random.default_rng(42)
injection_pool_raw  = make_injection_pool(train_raw, rng=rng_global)

print()
print('Building features for each motor injection pool ...')
injection_pool_feat = {}
for mid, raw_pool in injection_pool_raw.items():
    print(f'  Motor {mid} ({len(raw_pool)} rows, {raw_pool["test_condition"].nunique()} sequences):')
    feat_pool = build_features_for_df(raw_pool)
    injection_pool_feat[mid] = feat_pool
    print(f'    -> features built, shape: {feat_pool.shape}')

print('\nInjection pool ready.')

Building raw injection pool from training data ...
  Motor 1: 4 synthetic sequences, 1043 fault rows
  Motor 2: 4 synthetic sequences, 1041 fault rows
  Motor 3: 16 synthetic sequences, 3704 fault rows
  Motor 4: 4 synthetic sequences, 991 fault rows
  Motor 5: 16 synthetic sequences, 2752 fault rows
  Motor 6: 4 synthetic sequences, 811 fault rows

Building features for each motor injection pool ...
  Motor 1 (6132 rows, 4 sequences):
  Building features for 4 sequences ...
  ... 4/4
    -> features built, shape: (6132, 173)
  Motor 2 (5533 rows, 4 sequences):
  Building features for 4 sequences ...
  ... 4/4
    -> features built, shape: (5533, 173)
  Motor 3 (12519 rows, 16 sequences):
  Building features for 16 sequences ...
  ... 10/16
  ... 16/16
    -> features built, shape: (12519, 173)
  Motor 4 (7637 rows, 4 sequences):
  Building features for 4 sequences ...
  ... 4/4
    -> features built, shape: (7637, 173)
  Motor 5 (17140 rows, 16 sequences):
  Building features for 16 s

## § 8 · `train_motor()` function

Full training pipeline for one motor:

1. Identify feature columns for this motor from `train_feat_df`.
2. Stack original training rows + injection pool rows for this motor.
3. **GroupKFold(5)** by test_condition → OOF predictions.
4. Find OOF-optimal rate via `find_oof_optimal_rate`.
5. Compute prevalence in trust-filtered additional_data.
6. Determine final `target_rate` (protected rate or capped by additional prevalence).
7. Refit on the full training pool.
8. Predict test probabilities.
9. For episode motors (2, 6): use `decode_to_rate` (Viterbi); else `rate_matched_threshold`.
10. Store results in `models[motor_id]`.

In [12]:
def _get_motor_feature_cols(df: pd.DataFrame, motor_id: int) -> list:
    """
    Return all engineered feature column names for motor_id,
    excluding label columns and metadata columns.
    """
    prefix      = f'data_motor_{motor_id}_'
    label_col   = f'data_motor_{motor_id}_label'
    cross_motor = [
        f'data_motor_{motor_id}_volt_dev_peers',
        f'data_motor_{motor_id}_other_vel_sum',
        f'data_motor_{motor_id}_other_moving_count',
    ]
    cols = [
        c for c in df.columns
        if c.startswith(prefix)
        and c != label_col
        and not c.endswith('_label')   # guard for multi-motor label columns
    ]
    # Also include mean_voltage as a global feature
    if 'mean_voltage' in df.columns:
        cols.append('mean_voltage')
    return sorted(set(cols))


def train_motor(motor_id: int):
    SEP  = '=' * 70
    LINE = '-' * 70
    print(f'\n{SEP}')
    print(f'  MOTOR {motor_id}   algo={FINAL_MODELS[motor_id]}')
    print(f'{SEP}')

    label_col = f'data_motor_{motor_id}_label'
    algo      = FINAL_MODELS[motor_id]

    # ── Feature columns ───────────────────────────────────────────────────
    feat_cols = _get_motor_feature_cols(train_feat_df, motor_id)
    if not feat_cols:
        print('  SKIP — no feature columns found')
        return
    if label_col not in train_feat_df.columns:
        print('  SKIP — label column not found')
        return

    print(f'  Feature columns: {len(feat_cols)}')

    # ── Build training pool: original + injection ─────────────────────────
    base_pool = train_feat_df.dropna(subset=[label_col]).copy()

    pool_parts = [base_pool]
    if motor_id in injection_pool_feat:
        inj = injection_pool_feat[motor_id]
        # Align feature columns — injection may have fewer cols if extra were added
        missing_in_inj = [c for c in feat_cols if c not in inj.columns]
        for c in missing_in_inj:
            inj[c] = 0.0
        pool_parts.append(inj)
        print(f'  Injection pool: {len(inj)} rows, '
              f'{inj["test_condition"].nunique()} sequences')

    full_pool = pd.concat(pool_parts, ignore_index=True)
    full_pool[feat_cols] = full_pool[feat_cols].fillna(0)

    X_pool = full_pool[feat_cols].values.astype(np.float32)
    y_pool = full_pool[label_col].values.astype(int)
    g_pool = full_pool['test_condition'].values

    n0 = (y_pool == 0).sum()
    n1 = (y_pool == 1).sum()
    print(f'  Training pool: {len(y_pool)} rows  '
          f'(normal={n0}, fault={n1}, {100*n1/len(y_pool):.1f}%)')

    # ── GroupKFold OOF ────────────────────────────────────────────────────
    print(f'\n  GroupKFold OOF (5 folds) ...')
    gkf       = GroupKFold(n_splits=5)
    oof_probs  = np.zeros(len(y_pool), dtype=np.float32)
    oof_labels = y_pool.copy()

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_pool, y_pool, groups=g_pool)):
        X_tr, X_va = X_pool[tr_idx], X_pool[va_idx]
        y_tr, y_va = y_pool[tr_idx], y_pool[va_idx]

        sw_tr = compute_weights(y_tr)
        mdl   = _make_model(algo)

        # HistGradientBoosting does not accept sample_weight in fit for some versions;
        # pass if supported, else fit without
        try:
            mdl.fit(X_tr, y_tr, sample_weight=sw_tr)
        except TypeError:
            mdl.fit(X_tr, y_tr)

        probs_va         = mdl.predict_proba(X_va)[:, 1]
        oof_probs[va_idx] = probs_va

        # Quick fold F1 at 0.5 for info
        pred_va = (probs_va >= 0.5).astype(int)
        f1_fold = f1_score(y_va, pred_va, pos_label=1, zero_division=0)
        print(f'    Fold {fold+1}: val F1@0.5={f1_fold:.4f}  '
              f'(fault rows in val: {y_va.sum()})')

    # ── OOF optimal rate ──────────────────────────────────────────────────
    oof_optimal_rate = find_oof_optimal_rate(oof_probs, oof_labels)
    oof_thr          = rate_matched_threshold(oof_probs, oof_optimal_rate)
    oof_pred         = (oof_probs >= oof_thr).astype(int)
    oof_f1           = f1_score(oof_labels, oof_pred, pos_label=1, zero_division=0)
    print(f'\n  OOF optimal rate: {oof_optimal_rate:.4f}  '
          f'thr={oof_thr:.4f}  OOF F1={oof_f1:.4f}')

    # ── Additional-data prevalence ────────────────────────────────────────
    if len(additional_df) > 0:
        add_prevalence = compute_additional_prevalence(additional_df, motor_id)
    else:
        add_prevalence = n1 / max(1, len(y_pool))
    print(f'  Additional-data prevalence (motor {motor_id}): {add_prevalence:.4f}')

    # ── Target rate ───────────────────────────────────────────────────────
    if motor_id in PROTECTED_RATES:
        target_rate = PROTECTED_RATES[motor_id]
        print(f'  Using PROTECTED rate: {target_rate:.4f}')
    else:
        cap         = TEST_RATE_CAP_MULT * add_prevalence
        target_rate = min(oof_optimal_rate, cap)
        print(f'  OOF rate={oof_optimal_rate:.4f}, cap (2x prev)={cap:.4f}  '
              f'-> target_rate={target_rate:.4f}')

    # ── Refit final model on full training pool ───────────────────────────
    print(f'\n  Fitting final model on full pool ({len(y_pool)} rows) ...')
    sw_full    = compute_weights(y_pool)
    final_mdl  = _make_model(algo)
    try:
        final_mdl.fit(X_pool, y_pool, sample_weight=sw_full)
    except TypeError:
        final_mdl.fit(X_pool, y_pool)

    # ── Test predictions ──────────────────────────────────────────────────
    missing_in_test = [c for c in feat_cols if c not in test_feat_df.columns]
    for c in missing_in_test:
        test_feat_df[c] = 0.0

    X_test    = test_feat_df[feat_cols].fillna(0).values.astype(np.float32)
    test_probs = final_mdl.predict_proba(X_test)[:, 1]

    if motor_id in EPISODE_MOTORS:
        # Viterbi episode decoding
        print(f'  Viterbi episode decoding (motor {motor_id} in EPISODE_MOTORS) ...')
        # Estimate transition probabilities from training labels
        train_labels_seq = base_pool[label_col].values
        ep_len  = mean_episode_length(train_labels_seq)
        a11     = float(np.clip(1.0 - 1.0 / max(ep_len, 2), 0.90, 0.999))
        # Estimate stay-normal from fraction of normal rows
        p_fault = float(train_labels_seq.mean())
        a00     = float(np.clip(1.0 - p_fault / max(1 - p_fault, 1e-6) / max(ep_len, 1),
                                0.990, 0.9999))
        print(f'    a11={a11:.4f}  a00={a00:.4f}  mean_ep_len={ep_len:.1f}')

        seq_ids_test = test_feat_df['test_condition'].values
        test_preds   = decode_to_rate(test_probs, seq_ids_test, target_rate, a11, a00)
    else:
        thr_test   = rate_matched_threshold(test_probs, target_rate)
        test_preds = (test_probs >= thr_test).astype(int)
        print(f'  Rate-matched threshold: {thr_test:.4f}')

    pct_pos = 100.0 * test_preds.mean()
    print(f'  Test predictions: {test_preds.sum()} positive ({pct_pos:.2f}%)')

    # ── Store results ─────────────────────────────────────────────────────
    models[motor_id] = {
        'model'       : final_mdl,
        'algo'        : algo,
        'feat_cols'   : feat_cols,
        'target_rate' : target_rate,
        'oof_rate'    : oof_optimal_rate,
        'val_f1'      : oof_f1,
        'test_preds'  : test_preds,
        'test_probs'  : test_probs,
    }

    print(f'\n{SEP}')
    print(f'  Motor {motor_id} done.  OOF F1={oof_f1:.4f}  rate={target_rate:.4f}')
    print(f'{SEP}')


print('train_motor() defined')

train_motor() defined


## § 9 · Motor 1

Algorithm: RandomForest.  Shape features included.

In [13]:
train_motor(1)


  MOTOR 1   algo=RF
  Feature columns: 29
  Injection pool: 6132 rows, 4 sequences
  Training pool: 105539 rows  (normal=100681, fault=4858, 4.6%)

  GroupKFold OOF (5 folds) ...
    Fold 1: val F1@0.5=0.1320  (fault rows in val: 1495)
    Fold 2: val F1@0.5=0.1480  (fault rows in val: 394)
    Fold 3: val F1@0.5=0.1247  (fault rows in val: 752)
    Fold 4: val F1@0.5=0.0320  (fault rows in val: 980)
    Fold 5: val F1@0.5=0.0000  (fault rows in val: 1237)

  OOF optimal rate: 0.1762  thr=0.0644  OOF F1=0.2918
  Additional-data prevalence (motor 1): 0.0410
  OOF rate=0.1762, cap (2x prev)=0.0821  -> target_rate=0.0821

  Fitting final model on full pool (105539 rows) ...
  Rate-matched threshold: 0.2066
  Test predictions: 1163 positive (8.22%)

  Motor 1 done.  OOF F1=0.2918  rate=0.0821


## § 10 · Motor 2

Algorithm: RandomForest.  Shape features included.  Viterbi episode decoding applied.

In [14]:
train_motor(2)


  MOTOR 2   algo=RF
  Feature columns: 29
  Injection pool: 5533 rows, 4 sequences
  Training pool: 104940 rows  (normal=95803, fault=9137, 8.7%)

  GroupKFold OOF (5 folds) ...
    Fold 1: val F1@0.5=0.0898  (fault rows in val: 7407)
    Fold 2: val F1@0.5=0.6528  (fault rows in val: 324)
    Fold 3: val F1@0.5=0.4382  (fault rows in val: 369)
    Fold 4: val F1@0.5=0.5317  (fault rows in val: 299)
    Fold 5: val F1@0.5=0.5248  (fault rows in val: 738)

  OOF optimal rate: 0.0493  thr=0.3628  OOF F1=0.2649
  Additional-data prevalence (motor 2): 0.0227
  OOF rate=0.0493, cap (2x prev)=0.0454  -> target_rate=0.0454

  Fitting final model on full pool (104940 rows) ...
  Viterbi episode decoding (motor 2 in EPISODE_MOTORS) ...
    a11=0.9990  a00=0.9999  mean_ep_len=1349.3
  Test predictions: 527 positive (3.72%)

  Motor 2 done.  OOF F1=0.2649  rate=0.0454


## § 11 · Motor 3

Algorithm: RandomForest.  Shape features included.  Protected rate = 0.005.

In [15]:
train_motor(3)


  MOTOR 3   algo=RF
  Feature columns: 29
  Injection pool: 12519 rows, 16 sequences
  Training pool: 111926 rows  (normal=107388, fault=4538, 4.1%)

  GroupKFold OOF (5 folds) ...
    Fold 1: val F1@0.5=0.6853  (fault rows in val: 815)
    Fold 2: val F1@0.5=0.6187  (fault rows in val: 1342)
    Fold 3: val F1@0.5=0.9379  (fault rows in val: 462)
    Fold 4: val F1@0.5=0.3538  (fault rows in val: 1156)
    Fold 5: val F1@0.5=0.9011  (fault rows in val: 763)

  OOF optimal rate: 0.0403  thr=0.3069  OOF F1=0.7657
  Additional-data prevalence (motor 3): 0.0118
  Using PROTECTED rate: 0.0050

  Fitting final model on full pool (111926 rows) ...
  Rate-matched threshold: 0.3462
  Test predictions: 71 positive (0.50%)

  Motor 3 done.  OOF F1=0.7657  rate=0.0050


## § 12 · Motor 4

Algorithm: RandomForest.  No shape features.

In [16]:
train_motor(4)


  MOTOR 4   algo=RF
  Feature columns: 27
  Injection pool: 7637 rows, 4 sequences
  Training pool: 107044 rows  (normal=97379, fault=9665, 9.0%)

  GroupKFold OOF (5 folds) ...
    Fold 1: val F1@0.5=0.0195  (fault rows in val: 6739)
    Fold 2: val F1@0.5=0.7739  (fault rows in val: 558)
    Fold 3: val F1@0.5=0.0346  (fault rows in val: 334)
    Fold 4: val F1@0.5=0.2069  (fault rows in val: 1846)
    Fold 5: val F1@0.5=0.7375  (fault rows in val: 188)

  OOF optimal rate: 0.2124  thr=0.0782  OOF F1=0.2982
  Additional-data prevalence (motor 4): 0.0322
  OOF rate=0.2124, cap (2x prev)=0.0644  -> target_rate=0.0644

  Fitting final model on full pool (107044 rows) ...
  Rate-matched threshold: 0.1590
  Test predictions: 912 positive (6.44%)

  Motor 4 done.  OOF F1=0.2982  rate=0.0644


## § 13 · Motor 5

Algorithm: HistGradientBoosting.  Shape features included.  Protected rate = 0.004.

In [17]:
train_motor(5)


  MOTOR 5   algo=HGB
  Feature columns: 29
  Injection pool: 17140 rows, 16 sequences
  Training pool: 116547 rows  (normal=112173, fault=4374, 3.8%)

  GroupKFold OOF (5 folds) ...
    Fold 1: val F1@0.5=0.8299  (fault rows in val: 734)
    Fold 2: val F1@0.5=0.5994  (fault rows in val: 1441)
    Fold 3: val F1@0.5=0.9605  (fault rows in val: 549)
    Fold 4: val F1@0.5=0.5556  (fault rows in val: 1196)
    Fold 5: val F1@0.5=0.8260  (fault rows in val: 454)

  OOF optimal rate: 0.0252  thr=0.6677  OOF F1=0.7491
  Additional-data prevalence (motor 5): 0.0239
  Using PROTECTED rate: 0.0040

  Fitting final model on full pool (116547 rows) ...
  Rate-matched threshold: 0.6032
  Test predictions: 57 positive (0.40%)

  Motor 5 done.  OOF F1=0.7491  rate=0.0040


## § 14 · Motor 6

Algorithm: HistGradientBoosting.  No shape features.  Viterbi episode decoding + protected rate = 0.008.

In [18]:
train_motor(6)


  MOTOR 6   algo=HGB
  Feature columns: 27
  Injection pool: 10860 rows, 4 sequences
  Training pool: 110267 rows  (normal=106499, fault=3768, 3.4%)

  GroupKFold OOF (5 folds) ...
    Fold 1: val F1@0.5=0.1239  (fault rows in val: 960)
    Fold 2: val F1@0.5=0.5498  (fault rows in val: 128)
    Fold 3: val F1@0.5=0.4414  (fault rows in val: 1686)
    Fold 4: val F1@0.5=0.6440  (fault rows in val: 912)
    Fold 5: val F1@0.5=0.3478  (fault rows in val: 82)

  OOF optimal rate: 0.0282  thr=0.1327  OOF F1=0.4561
  Additional-data prevalence (motor 6): 0.0171
  Using PROTECTED rate: 0.0080

  Fitting final model on full pool (110267 rows) ...
  Viterbi episode decoding (motor 6 in EPISODE_MOTORS) ...
    a11=0.9963  a00=0.9999  mean_ep_len=268.8
  Test predictions: 114 positive (0.81%)

  Motor 6 done.  OOF F1=0.4561  rate=0.0080


## § 15 · Summary

Shows model type, target rate, OOF F1 per motor, and a text bar chart.

In [19]:
SEP  = '=' * 70
LINE = '-' * 70

print(SEP)
print('  FINAL SUMMARY — v3 Pipeline')
print(SEP)
print(f'{"Motor":<8} {"Algo":<6} {"OOF F1":>8} {"Rate":>8} {"#Test+":>8}   Bar')
print(LINE)

oof_f1_scores = {}
for mid in range(1, 7):
    info = models.get(mid)
    if info is None:
        print(f'{mid:<8} ---    not trained')
        continue
    algo  = info['algo']
    f1    = info['val_f1']
    rate  = info['target_rate']
    n_pos = int(info['test_preds'].sum())
    bar   = '#' * int(f1 * 40)
    viterbi_mark = ' (Viterbi)' if mid in EPISODE_MOTORS else ''
    print(f'{mid:<8} {algo:<6} {f1:>8.4f} {rate:>8.4f} {n_pos:>8}   {bar}{viterbi_mark}')
    oof_f1_scores[mid] = f1

print(LINE)
if oof_f1_scores:
    mean_f1 = sum(oof_f1_scores.values()) / len(oof_f1_scores)
    print(f'  Global mean OOF F1 ({len(oof_f1_scores)} motors): {mean_f1:.4f}')
print(SEP)

  FINAL SUMMARY — v3 Pipeline
Motor    Algo     OOF F1     Rate   #Test+   Bar
----------------------------------------------------------------------
1        RF       0.2918   0.0821     1163   ###########
2        RF       0.2649   0.0454      527   ########## (Viterbi)
3        RF       0.7657   0.0050       71   ##############################
4        RF       0.2982   0.0644      912   ###########
5        HGB      0.7491   0.0040       57   #############################
6        HGB      0.4561   0.0080      114   ################## (Viterbi)
----------------------------------------------------------------------
  Global mean OOF F1 (6 motors): 0.4710


## § 16 · Submission

Assembles the final submission CSV from per-motor test predictions.

`EXCLUDE_MOTOR = N` → forces that motor's predictions to `-1` (Kaggle trick to avoid spending a real submission).
`EXCLUDE_MOTOR = -1` → full submission (all motors).

In [20]:
# Set to a motor number (1-6) to exclude it with -1 labels (Kaggle trick)
# Set to -1 for a full submission with all motors
EXCLUDE_MOTOR = -1

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
final_sub  = sample_sub.copy()

# Build per-sequence test predictions lookup
# test_feat_df has one row per timestep; we need to align with sample_sub
for mid in range(1, 7):
    label_out = f'data_motor_{mid}_label'
    if mid not in models:
        print(f'Motor {mid}: no model — filling with 0')
        final_sub[label_out] = 0
        continue

    test_preds_arr = models[mid]['test_preds']
    # test_feat_df rows are in the same order as test_raw was loaded
    # We need to map predictions into sample_sub rows via test_condition
    # Build a mapping: (test_condition, row_within_sequence) -> pred
    test_conds_ordered = test_feat_df['test_condition'].values

    # Create a temporary DataFrame with test_condition + predictions
    preds_df = test_feat_df[['test_condition', 'time']].copy()
    preds_df['_pred'] = test_preds_arr

    # For each test_condition in sample_sub, match rows by position within sequence
    for seq in sample_sub['test_condition'].unique():
        sub_mask  = (sample_sub['test_condition'] == seq)
        pred_mask = (preds_df['test_condition'] == seq)
        sub_idx   = final_sub.index[sub_mask]
        pred_seq  = preds_df.loc[pred_mask, '_pred'].values
        exp_len   = int(sub_mask.sum())

        if len(pred_seq) == 0:
            final_sub.loc[sub_mask, label_out] = 0
            continue

        if len(pred_seq) != exp_len:
            # Length mismatch: pad or truncate
            if len(pred_seq) > exp_len:
                pred_seq = pred_seq[:exp_len]
            else:
                pred_seq = np.pad(pred_seq, (0, exp_len - len(pred_seq)), 'constant')

        final_sub.loc[sub_mask, label_out] = pred_seq

# Cast to int
for mid in range(1, 7):
    col = f'data_motor_{mid}_label'
    final_sub[col] = final_sub[col].fillna(0).astype(int)

# Kaggle exclude-motor trick
if EXCLUDE_MOTOR in range(1, 7):
    print(f'Kaggle trick: motor {EXCLUDE_MOTOR} -> -1')
    final_sub[f'data_motor_{EXCLUDE_MOTOR}_label'] = -1
    out_path = os.path.join(SUBMISSION_DIR, f'motor_excluded_{EXCLUDE_MOTOR}_v3_submission.csv')
else:
    out_path = os.path.join(SUBMISSION_DIR, 'motor_submission_v3.csv')

final_sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {final_sub.shape}')
print()

# Summary of test predictions
print(f'{"Motor":<8} {"Algo":<6} {"0 normal":>10} {"1 fault":>10} {"% fault":>9}')
print('-' * 50)
for mid in range(1, 7):
    col  = f'data_motor_{mid}_label'
    algo = models.get(mid, {}).get('algo', '?')
    c0   = (final_sub[col] == 0).sum()
    c1   = (final_sub[col] == 1).sum()
    cn   = (final_sub[col] == -1).sum()  # excluded
    if cn > 0:
        print(f'{mid:<8} {algo:<6} {"EXCLUDED (-1)":>22}')
    else:
        pct = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
        print(f'{mid:<8} {algo:<6} {c0:>10} {c1:>10} {pct:>8.2f}%')

final_sub.head()

Saved: c:\Users\oscar\Desktop\TDS INDUSTRY\motor_submission_v3.csv
Shape: (14157, 8)

Motor    Algo     0 normal    1 fault   % fault
--------------------------------------------------
1        RF          12994       1163     8.22%
2        RF          13630        527     3.72%
3        RF          14086         71     0.50%
4        RF          13245        912     6.44%
5        HGB         14100         57     0.40%
6        HGB         14043        114     0.81%


,idx,data_motor_1_label,data_motor_2_label,data_motor_3_label,data_motor_4_label,data_motor_5_label,data_motor_6_label,test_condition
0,0,0,0,0,0,0,0,20240527_094865
1,1,0,0,0,0,0,0,20240527_094865
2,2,0,0,0,0,0,0,20240527_094865
3,3,0,0,0,0,0,0,20240527_094865
4,4,0,0,0,0,0,0,20240527_094865
